### Rebuilds `openalex.works.work_authorships` in full each run (Walden End to End workflow)

Full rebuild every run (oxjob #660): reads the current work_authors / affiliation-MV /
profile state for every work, so enrichment staleness self-corrects nightly by
construction. Replaces the former watermark + change-detection intake, the
MV-institution bolt-on, and the oxjob #582 clear-empty step (subsumed by the
empty-row branch). ~6 min serverless XL measured for the full build.

### Step 1: Create enriched authors

In [ ]:
REFRESH MATERIALIZED VIEW openalex.works.work_author_affiliations_mv

In [ ]:
%run ../utils/variables

In [ ]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.work_authorships')
CLUSTER BY (work_id) AS (
WITH base_works AS (
    -- Full rebuild (oxjob #660): every work with authorships, every run. No watermark,
    -- no change-detection, no per-field bolt-ons — enrichment staleness (bindings,
    -- institutions, affiliation matches, profile names/orcids) self-corrects nightly.
    SELECT
        id AS work_id,
        authorships,
        updated_date AS updated_datetime
    FROM identifier('openalex' || :env_suffix || '.works.openalex_works_base')
    WHERE authorships IS NOT NULL
      AND SIZE(authorships) > 0
),
institution_lineage AS (
    SELECT
        institution_id,
        FILTER(lineage_ids, id -> NOT ARRAY_CONTAINS(SUPER_SYSTEM_INSTITUTIONS, id)) AS lineage_ids
    FROM openalex.institutions.institution_ancestors
),
-- 1. Get Institution Details (Grouped by Work/Seq)
author_institutions_with_details AS (
    SELECT
        ai.work_id,
        ai.author_sequence,
        ARRAY_SORT(ARRAY_DISTINCT(FLATTEN(COLLECT_SET(ai.raw_countries)))) AS raw_parsed_countries,
        ARRAY_SORT(
          FILTER(
            COLLECT_SET(
              CASE WHEN inst.id IS NOT NULL THEN
                STRUCT(
                    inst.iso3166_code AS country_code,
                    inst.display_name,
                    CONCAT('https://openalex.org/I', ai.institution_id) AS id,
                    ARRAY_SORT(
                        TRANSFORM(
                            ARRAY_COMPACT(CONCAT(ARRAY(ai.institution_id), COALESCE(il.lineage_ids, ARRAY()))),
                            id -> CONCAT('https://openalex.org/I', id)
                        )
                    ) AS lineage,
                    CASE
                        WHEN inst.ror_id IS NULL THEN NULL
                        WHEN inst.ror_id LIKE 'https://ror.org/%' THEN inst.ror_id
                        ELSE CONCAT('https://ror.org/', inst.ror_id)
                    END AS ror,
                    inst.type
                )
              END
            ),
            x -> x IS NOT NULL
          ),
            (left, right) -> CASE
                WHEN left.id < right.id THEN -1
                WHEN left.id > right.id THEN 1
                ELSE 0
            END
        ) AS institutions
    FROM (
        SELECT
            work_id,
            author_sequence,
            raw_countries,
            EXPLODE(institution_ids) AS institution_id
        FROM identifier('openalex' || :env_suffix || '.works.work_author_affiliations_mv')
        WHERE institution_ids IS NOT NULL AND SIZE(institution_ids) > 0
    ) ai
    LEFT JOIN openalex.institutions.institutions inst ON inst.id = ai.institution_id
    LEFT JOIN institution_lineage il ON il.institution_id = ai.institution_id
    GROUP BY ai.work_id, ai.author_sequence
),
-- 2. Get Author IDs (Grouped by Work/Seq)
author_id_lookup AS (
    SELECT 
        work_id, 
        author_sequence, 
        MAX(author_id) as author_id
    FROM identifier('openalex' || :env_suffix || '.works.work_author_affiliations_mv')
    GROUP BY work_id, author_sequence
),
-- 3. Enrich Author IDs with Profile Data (OpenAlex Authors + Profiles)
author_data_enriched AS (
    SELECT 
        ail.work_id,
        ail.author_sequence,
        ail.author_id,
        -- Priority 1: Existing OpenAlex Author
        -- Priority 2: Newly Minted Author (from Profiles)
        COALESCE(oa.display_name, ar.display_name) as best_display_name,
        -- ORCID curation-aware: prefer curated openalex_authors.orcid (override from
        -- CreateAuthors); fall back to organic authors.orcid only for newly-minted authors
        -- not yet in openalex_authors. Mirrors best_display_name. (oxjob #410)
        CASE WHEN oa.id IS NOT NULL THEN oa.orcid ELSE ar.orcid END as best_orcid
    FROM author_id_lookup ail
    -- Join to Main Table (Existing Authors)
    LEFT JOIN openalex.authors.openalex_authors oa 
        ON ail.author_id = oa.id
    -- Join to Profiles (New Authors)
    LEFT JOIN openalex.authors.authors ar 
        ON ail.author_id = ar.id
),
affiliations_map_ids AS (
    SELECT
        wam.work_id,
        wam.author_sequence,
        wam.raw_affiliation_string,
        -- ARRAY_SORT: nightly full rebuild requires deterministic output, else unchanged
        -- works churn the enriched content hash every run
        ARRAY_SORT(COLLECT_LIST(CONCAT('https://openalex.org/I', inst.id))) AS institution_ids
    FROM (
        SELECT work_id, author_sequence, raw_affiliation_string,
               EXPLODE_OUTER(institution_ids) AS institution_id
        FROM identifier('openalex' || :env_suffix || '.works.work_author_affiliations_mv')
        WHERE raw_affiliation_string IS NOT NULL
    ) wam
    LEFT JOIN openalex.institutions.institutions inst ON inst.id = wam.institution_id
    GROUP BY wam.work_id, wam.author_sequence, wam.raw_affiliation_string
),
affiliations_map AS (
    SELECT
        work_id,
        author_sequence,
        MAP_FROM_ENTRIES(
            ARRAY_DISTINCT(
                COLLECT_LIST(NAMED_STRUCT('key', raw_affiliation_string, 'value', institution_ids))
            )
        ) AS aff_map
    FROM affiliations_map_ids
    GROUP BY work_id, author_sequence
),
-- 4. Build Final Lookup Map
author_institution_lookup AS (
    SELECT
        ade.work_id,
        MAP_FROM_ENTRIES(
            COLLECT_LIST(
                STRUCT(
                    ade.author_sequence,
                    STRUCT(
                        -- Enriched Author Data
                        ade.author_id,
                        ade.best_display_name,
                        ade.best_orcid,
                        
                        -- Institution Data
                        details.institutions,
                        details.raw_parsed_countries,
                        am.aff_map
                    )
                )
            )
        ) AS author_lookup
    FROM author_data_enriched ade
    LEFT JOIN author_institutions_with_details details
        ON ade.work_id = details.work_id 
        AND ade.author_sequence = details.author_sequence
    LEFT JOIN affiliations_map am 
        ON ade.work_id = am.work_id 
        AND ade.author_sequence = am.author_sequence
    GROUP BY ade.work_id
),
-- 5. Build enriched authorships array
enriched_authorships AS (
    SELECT
        ba.work_id,
        ba.updated_datetime,
        TRANSFORM(
            ba.authorships,
            (auth, idx) -> STRUCT(
                TRANSFORM(
                    COALESCE(auth.raw_affiliation_strings, ARRAY()),
                    s -> STRUCT(
                        COALESCE(ELEMENT_AT(ELEMENT_AT(ail.author_lookup, idx).aff_map, s), ARRAY()) AS institution_ids,
                        s AS raw_affiliation_string
                    )
                ) AS affiliations,
                
                STRUCT(
                    CASE 
                       WHEN ELEMENT_AT(ail.author_lookup, idx).author_id IS NOT NULL 
                       THEN CONCAT('https://openalex.org/A', CAST(ELEMENT_AT(ail.author_lookup, idx).author_id AS STRING))
                       ELSE auth.author.id 
                    END as id,
                    -- Use Display Name from OA/Profiles, fallback to raw work data
                    COALESCE(
                        ELEMENT_AT(ail.author_lookup, idx).best_display_name, 
                        auth.author.display_name
                    ) as display_name,
                    -- Use ORCID from OA, fallback to raw work data
                    CASE 
                        WHEN ELEMENT_AT(ail.author_lookup, idx).author_id IS NOT NULL 
                        THEN ELEMENT_AT(ail.author_lookup, idx).best_orcid
                        ELSE auth.raw_orcid
                    END as orcid
                ) as author,

                -- Source-level ORCID from crossref/datacite (before profile enrichment)
                auth.raw_orcid AS raw_orcid,
                auth.author_position,
                auth.author_order_number,
                CASE
                    WHEN ELEMENT_AT(ail.author_lookup, idx).institutions IS NOT NULL 
                         AND SIZE(FILTER(ELEMENT_AT(ail.author_lookup, idx).institutions.country_code, c -> c IS NOT NULL AND c <> '')) > 0
                        THEN ARRAY_SORT(ARRAY_DISTINCT(FILTER(ELEMENT_AT(ail.author_lookup, idx).institutions.country_code, c -> c IS NOT NULL AND c <> '')))
                    WHEN ELEMENT_AT(ail.author_lookup, idx).raw_parsed_countries IS NOT NULL
                        THEN ARRAY_SORT(ELEMENT_AT(ail.author_lookup, idx).raw_parsed_countries)
                    ELSE ARRAY()
                END AS countries,
                COALESCE(ELEMENT_AT(ail.author_lookup, idx).institutions, ARRAY()) AS institutions,
                auth.is_corresponding,
                auth.raw_affiliation_strings,
                auth.raw_author_name
            )
        ) AS authorships
    FROM base_works ba
    LEFT JOIN author_institution_lookup ail ON ba.work_id = ail.work_id
)
-- 6. Compute corresponding ids, with guarded single-institution fallback (oxjob #517).
-- corresponding_institution_ids: organic value (institutions of is_corresponding authors)
-- when present; else, when EVERY author has a non-empty raw affiliation string AND a
-- linked institution AND there is exactly one distinct institution across all authors,
-- fall back to that single institution. Institution-level only: corresponding_author_ids
-- is left untouched (we know WHERE, not WHO).
, corr AS (
    SELECT
        work_id,
        updated_datetime,
        authorships,
        SIZE(authorships) AS authors_count,
        ARRAY_SORT(ARRAY_DISTINCT(
            ARRAY_COMPACT(
                TRANSFORM(
                    FILTER(authorships, a -> a.is_corresponding = true),
                    a -> a.author.id
                )
            )
        )) AS corresponding_author_ids,
        -- organic: institutions of authors explicitly flagged is_corresponding
        ARRAY_SORT(ARRAY_DISTINCT(
            ARRAY_COMPACT(
                FLATTEN(
                    TRANSFORM(
                        FILTER(authorships, a -> a.is_corresponding = true),
                        a -> TRANSFORM(a.institutions, i -> i.id)
                    )
                )
            )
        )) AS organic_corr_inst,
        -- distinct institutions across ALL authors
        ARRAY_SORT(ARRAY_DISTINCT(
            ARRAY_COMPACT(
                FLATTEN(TRANSFORM(authorships, a -> TRANSFORM(a.institutions, i -> i.id)))
            )
        )) AS all_distinct_inst,
        -- guardrail: count of authors lacking any non-empty raw affiliation string
        SIZE(FILTER(authorships, a ->
            SIZE(FILTER(COALESCE(a.raw_affiliation_strings, ARRAY()), s -> s IS NOT NULL AND s <> '')) = 0
        )) AS n_missing_ras,
        -- guardrail: count of authors lacking any linked institution
        SIZE(FILTER(authorships, a -> SIZE(COALESCE(a.institutions, ARRAY())) = 0)) AS n_missing_inst
    FROM enriched_authorships
)
SELECT
    work_id,
    updated_datetime,
    authorships,
    authors_count,
    corresponding_author_ids,
    CASE
        WHEN SIZE(organic_corr_inst) > 0 THEN organic_corr_inst
        WHEN n_missing_ras = 0 AND n_missing_inst = 0 AND SIZE(all_distinct_inst) = 1
            THEN all_distinct_inst
        ELSE organic_corr_inst
    END AS corresponding_institution_ids,
    -- Counts over the first-5000 slice: identical semantics to the former
    -- CreateWorksEnriched computation (its authorships MERGE slices to 5000)
    COALESCE(SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(FLATTEN(SLICE(authorships, 1, 5000).institutions.id)))), 0) AS institutions_distinct_count,
    COALESCE(SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(FLATTEN(SLICE(authorships, 1, 5000).institutions.country_code)))), 0) AS countries_distinct_count
FROM corr

UNION ALL

-- Works whose base authorships went empty keep an empty row (oxjob #582): if the row
-- disappeared instead, CreateWorksEnriched's MERGE would never match and openalex_works
-- would serve the stale authorships forever. Works that regain authorships re-enter the
-- main branch automatically. Works absent from works_base entirely are dropped (they no
-- longer exist downstream: openalex_works is a clone of base). Self-read of
-- work_authorships inside its own CREATE OR REPLACE is safe: Delta pins the read
-- to the pre-replace snapshot.
SELECT
    wa.work_id,
    wb.updated_date AS updated_datetime,
    CAST(ARRAY() AS ARRAY<STRUCT<
        affiliations: ARRAY<STRUCT<institution_ids: ARRAY<STRING>, raw_affiliation_string: STRING>>,
        author: STRUCT<id: STRING, display_name: STRING, orcid: STRING>,
        raw_orcid: STRING,
        author_position: STRING,
        author_order_number: INT,
        countries: ARRAY<STRING>,
        institutions: ARRAY<STRUCT<country_code: STRING, display_name: STRING, id: STRING, lineage: ARRAY<STRING>, ror: STRING, type: STRING>>,
        is_corresponding: BOOLEAN,
        raw_affiliation_strings: ARRAY<STRING>,
        raw_author_name: STRING>>) AS authorships,
    0 AS authors_count,
    CAST(ARRAY() AS ARRAY<STRING>) AS corresponding_author_ids,
    CAST(ARRAY() AS ARRAY<STRING>) AS corresponding_institution_ids,
    0 AS institutions_distinct_count,
    0 AS countries_distinct_count
FROM identifier('openalex' || :env_suffix || '.works.work_authorships') wa
JOIN identifier('openalex' || :env_suffix || '.works.openalex_works_base') wb ON wb.id = wa.work_id
WHERE wb.authorships IS NULL OR SIZE(wb.authorships) = 0
);